# 05 · Validación y calidad de la medida

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/telmomm/nanovna-measurement-course/HEAD?labpath=es%2Fnotebooks%2F05_validacion_calidad.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/telmomm/nanovna-measurement-course/blob/main/es/notebooks/05_validacion_calidad.ipynb)

*Cuaderno avanzado.* En el cuaderno 03 calibraste medidas reales y conseguiste que una resistencia de 75 Ω leyera unos 74 Ω. ¿Es un buen resultado? «Parece correcto» no es una respuesta que puedas poner en un informe. Este cuaderno le hace tres preguntas a una medida antes de fiarse de ella, y después deja las respuestas por escrito de forma que otra persona pueda comprobarlas.

## Objetivos de aprendizaje

Al terminar este cuaderno serás capaz de:

- cuantificar la repetibilidad de una medida a partir de barridos repetidos y convertirla en una incertidumbre típica;
- usar la pasividad ($|\Gamma| \le 1$) como prueba física e interpretar una violación como evidencia sobre la medida;
- distinguir entre integridad de los datos, coherencia física y calidad de la medida;
- registrar el contexto y el historial de procesado de un resultado para que pueda trazarse y reproducirse.

**Necesitas:** el [cuaderno 03](03_calibracion_sol.ipynb). Sin hardware. Este cuaderno usa una biblioteca adicional, [`rfmeasurement`](https://github.com/telmomm/rfmeasurement), cuyos mensajes e informes están en inglés.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "vnacourse[validation] @ git+https://github.com/telmomm/nanovna-measurement-course"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import skrf as rf
from IPython.display import Markdown
from rfmeasurement.domain import (
    AnalysisResult,
    Measurand,
    Measurement,
    MeasurementContext,
    MetadataConfidence,
    ProvenanceRecord,
    ValidationResult,
    ValidationStatus,
)
from rfmeasurement.reporting import generate_report
from rfmeasurement.validation import DEFAULT_RULES, PassivityRule, ValidationRule, validate

import vnacourse
from vnacourse import calibration, plotting
from vnacourse.data import load_repetitions
from vnacourse.reflection import gamma_to_z

plotting.use_course_style()

## 1. Tres preguntas

Antes de usar una medida, pregúntate:

1. **¿Están íntegros los datos?** El número correcto de puntos, frecuencias ordenadas, sin valores ausentes ni infinitos. Un fallo aquí significa que el *fichero* está roto.
2. **¿Es físicamente plausible?** Un dispositivo pasivo no puede reflejar más potencia de la que recibe. Un fallo aquí significa que los *números* contradicen lo que supusimos sobre el dispositivo o el instrumento.
3. **¿Cuánto cambia si la repito?** Eso fija cuántas cifras del resultado significan algo.

Son preguntas de distinta naturaleza, y la diferencia importa. Un fichero roto hay que descartarlo. Un número físicamente inverosímil hay que *explicarlo*: te está diciendo algo. Empezamos por la tercera pregunta, porque su respuesta hace falta para juzgar la segunda.

In [ ]:
f_hz, short_reps = load_repetitions("standards", "short")
_, open_reps = load_repetitions("standards", "open")
_, load_reps = load_repetitions("standards", "load")
_, resistor_reps = load_repetitions("dut", "load_75ohm")
f_mhz = f_hz / 1e6

print(
    f"Estándares: {open_reps.shape[0]} repeticiones de cada uno, {len(f_hz)} puntos de frecuencia"
)
print(f"Resistencia de 75 Ω: {resistor_reps.shape[0]} repeticiones")

## 2. Repetibilidad

Cada estándar se midió diez veces sin cambiar nada a propósito. Si el instrumento fuera perfectamente repetible, los diez barridos serían idénticos. No lo son. A cada frecuencia resumimos su dispersión con una **desviación típica**. $\Gamma$ es complejo, así que combinamos la dispersión de su parte real y de su parte imaginaria:

$$s(f) = \sqrt{s^2_{\operatorname{Re}\Gamma}(f) + s^2_{\operatorname{Im}\Gamma}(f)}$$

> **Predice antes de ejecutar.** ¿Cuál de los tres estándares esperas que se repita mejor: el cortocircuito, el abierto o la carga? Piensa en lo que aprendiste sobre los términos de error en el cuaderno 02.

In [ ]:
def spread(repetitions):
    """Desviación típica de una magnitud compleja entre repeticiones, por frecuencia."""
    return np.sqrt(repetitions.real.var(axis=0, ddof=1) + repetitions.imag.var(axis=0, ddof=1))


spreads = {
    "cortocircuito": spread(short_reps),
    "abierto": spread(open_reps),
    "carga": spread(load_reps),
}

fig, ax = plt.subplots()
for name, s in spreads.items():
    ax.semilogy(f_mhz, s, label=name, linewidth=1.0)
    print(
        f"{name:<13} típica (mediana) {np.median(s):.4f}   "
        f"media {s.mean():.4f}   peor {s.max():.3f}"
    )
ax.set(xlabel="frecuencia (MHz)", ylabel="desviación típica de Γ (escala log.)")
ax.set_title("Dispersión de diez repeticiones de cada estándar")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)

La carga se repite unas seis veces mejor que el cortocircuito y el abierto. El motivo es el del cuaderno 02: todo lo que fluctúa en el camino de vuelta multiplica a $\Gamma$, así que se nota por completo en un dispositivo con $|\Gamma| = 1$ y apenas en uno con $\Gamma \approx 0$.

Una dispersión típica de 0.003 en una magnitud de tamaño 1 es un 0.3 %. Ese número es el suelo de todo lo demás: ninguna calibración calculada a partir de estos barridos puede ser fiable por debajo de él.

Fíjate en la diferencia entre la *mediana* y la *media*, y en el peor valor. La dispersión no es uniforme: hay picos, veinte veces por encima del nivel típico. ¿Dónde están?

In [ ]:
s = spreads["abierto"]
noisy = np.flatnonzero(s > 5 * np.median(s))
print(f"{len(noisy)} de {len(s)} puntos de frecuencia tienen más de 5 veces la dispersión típica")
print("Su posición en el barrido (índice del punto):")
print(noisy)

just_after = np.sum(noisy % 100 < 10)
print(f"\n{just_after} de ellos caen en los diez primeros puntos tras un múltiplo de 100")

Los puntos ruidosos no están repartidos al azar. La mayoría están en los primeros puntos tras los índices 100, 200, 300, etc., una décima parte del barrido. Un dispositivo no tiene ningún motivo para ser ruidoso cada cien puntos, así que este patrón pertenece a *cómo se adquirió el barrido*, no a lo que se midió. Es lo que verías si el instrumento recogiera los 1001 puntos en bloques de unos cien y necesitara un momento para estabilizarse al principio de cada bloque. Los datos por sí solos no pueden confirmar esa explicación, pero sí te dicen dónde mirar.

Dos consecuencias prácticas:

- **Usa estadística robusta para describir el comportamiento típico.** La mediana no se deja arrastrar por unos pocos picos; la media y el máximo sí.
- **Un pico en una repetición contamina el promedio de las diez.** El ejercicio 1 compara promediar con tomar la mediana entre repeticiones.

## 3. De la repetibilidad a la incertidumbre de un resultado

Toma un resultado concreto: **la resistencia de la resistencia de 75 Ω por debajo de 10 MHz**, definida como la media de $\operatorname{Re} Z$ en los puntos por debajo de 10 MHz, tras la calibración SOL. (Por debajo de 10 MHz el retardo del que hablamos en el cuaderno 03 gira la fase solo unos pocos grados, así que podemos leer la resistencia directamente.)

Se repitieron dos cosas, así que tenemos dos fuentes de dispersión:

- la propia resistencia se midió 3 veces;
- los estándares de calibración se midieron 10 veces, así que podemos calcular 10 calibraciones distintas.

Para una media de $n$ lecturas repetidas con desviación típica $s$, la **incertidumbre típica** de la media es $u = s/\sqrt{n}$. Una incertidumbre estimada así, a partir de la estadística de lecturas repetidas, se llama *de tipo A*.

In [ ]:
low = f_hz < 10e6


def resistance_below_10mhz(raw, terms):
    return gamma_to_z(calibration.correct(raw, terms)).real[low].mean()


all_standards = [reps.mean(axis=0) for reps in (short_reps, open_reps, load_reps)]
terms = calibration.solve_error_terms(all_standards)

# Fuente 1: las tres repeticiones de la resistencia, corregidas con la calibración completa.
r_per_sweep = np.array([resistance_below_10mhz(raw, terms) for raw in resistor_reps])

# Fuente 2: diez calibraciones, una por repetición de los estándares, aplicadas al barrido medio.
resistor_mean = resistor_reps.mean(axis=0)
r_per_calibration = np.array(
    [
        resistance_below_10mhz(
            resistor_mean,
            calibration.solve_error_terms([short_reps[i], open_reps[i], load_reps[i]]),
        )
        for i in range(len(open_reps))
    ]
)

u_dut = r_per_sweep.std(ddof=1) / np.sqrt(len(r_per_sweep))
u_cal = r_per_calibration.std(ddof=1) / np.sqrt(len(r_per_calibration))
u_combined = np.hypot(u_dut, u_cal)
resistance = resistance_below_10mhz(resistor_mean, terms)

print("Barridos de la resistencia:", np.round(r_per_sweep, 2), "Ω")
print("Calibraciones:             ", np.round(r_per_calibration, 2), "Ω")
print(f"\nu por la repetibilidad de la resistencia:  {u_dut:.2f} Ω")
print(f"u por la repetibilidad de la calibración:  {u_cal:.2f} Ω")
print(
    f"\nResistencia por debajo de 10 MHz: {resistance:.1f} Ω, "
    f"incertidumbre típica {u_combined:.1f} Ω"
)

Las dos contribuciones son independientes, así que se combinan como raíz de la suma de cuadrados. El resultado es de unos 74.0 Ω con una incertidumbre típica de unos 0.2 Ω.

El valor nominal es 75 Ω: a más de cinco incertidumbres típicas. La repetibilidad no puede explicar una diferencia tan grande, así que hay algo más. Los candidatos son todo lo que esta incertidumbre **no incluye**:

- la tolerancia de la propia resistencia (una pieza corriente del 1 % puede estar entre 74.25 Ω y 75.75 Ω; una del 5 %, entre 71 Ω y 79 Ω);
- los estándares de calibración, que supusimos ideales (cuaderno 03, sección 6): un error ahí desplaza todos los resultados en la misma dirección, siempre, así que repetir no lo revela;
- lo que haya entre el plano de calibración y la resistencia.

Este es el hábito más importante del cuaderno: una incertidumbre solo tiene sentido junto con la lista de lo que cubre. «74.0 ± 0.2 Ω» por sí solo afirma más de lo que sabemos. «74.0 Ω, con 0.2 Ω de incertidumbre típica por repetibilidad; estándares supuestos ideales» es honesto.

## 4. La pasividad como prueba física

Una resistencia, un cortocircuito y un abierto son pasivos: no tienen fuente de energía, así que no pueden devolver más de lo que reciben. Para un puerto eso significa

$$|\Gamma| \le 1 \quad\text{a todas las frecuencias.}$$

Es una prueba que no necesita ningún dispositivo de referencia ni una segunda medida: los datos la cumplen o no. Aplícala a tres medidas:

- el abierto en bruto, tal como lo mostró el instrumento;
- el abierto tras la calibración, con una calibración calculada a partir de las repeticiones 1–5 y aplicada a las repeticiones 6–10 (para que la prueba sea justa, como en el cuaderno 03);
- la resistencia de 75 Ω calibrada.

> **Predice antes de ejecutar.** Un abierto perfecto tiene $|\Gamma| = 1$ exactamente. Tras una buena calibración, con un poco de ruido restante, ¿qué fracción de los puntos esperas encontrar por encima de 1?

In [ ]:
terms_first_half = calibration.solve_error_terms(
    [reps[:5].mean(axis=0) for reps in (short_reps, open_reps, load_reps)]
)

cases = {
    "abierto, en bruto": open_reps.mean(axis=0),
    "abierto, calibrado": calibration.correct(open_reps[5:].mean(axis=0), terms_first_half),
    "resistencia de 75 Ω, calibrada": calibration.correct(resistor_mean, terms),
}

print(f"{'medida':<31} {'mayor |Γ|':>10} {'puntos > 1':>11} {'puntos > 1.01':>14}")
for name, gamma in cases.items():
    magnitude = np.abs(gamma)
    print(
        f"{name:<31} {magnitude.max():10.3f} {np.mean(magnitude > 1):11.0%} "
        f"{np.mean(magnitude > 1.01):14.1%}"
    )

Las tres cuentan historias distintas.

- **La resistencia** está lejos del límite. Se cumple la pasividad.
- **El abierto en bruto** está por encima de 1 en casi todos los puntos, y muy por encima en toda una banda. Es una violación real, y es *sistemática*: la misma en las diez repeticiones. No significa que el fichero esté corrupto. Significa que el seguimiento de reflexión del instrumento no es 1, que es exactamente lo que predijo el cuaderno 02 y lo que la calibración eliminó después. La violación era evidencia de un error sistemático sin corregir.
- **El abierto calibrado** está por encima de 1 en algo menos de la mitad de los puntos, por cantidades minúsculas. Mira cómo se distribuyen esas cantidades:

In [ ]:
excess = np.abs(cases["abierto, calibrado"]) - 1

fig, ax = plt.subplots()
ax.hist(excess, bins=np.linspace(-0.012, 0.012, 49), color=plotting.SERIES[0])
ax.axvline(0, color=plotting.INK, linewidth=1.0)
ax.set(xlabel="$|\\Gamma| - 1$", ylabel="número de puntos de frecuencia")
ax.set_title("Abierto calibrado: cuánto se aleja |Γ| de 1")

inside = np.mean(np.abs(excess) < 0.012)
print(f"Mediana de |Γ| − 1: {np.median(excess):+.4f}")
print(f"La mitad de los puntos está a menos de ±{np.median(np.abs(excess)):.4f} de 1")
print(f"El {1 - inside:.1%} de los puntos queda fuera de esta gráfica")

Los valores se dispersan a ambos lados de 1, en una banda estrecha: no más ancha de lo que la repetibilidad de la sección 2 te hace esperar para una media de cinco barridos. Así debe verse un dispositivo que lo refleja *todo*, medido con un poco de ruido: el ruido no sabe que 1 es un límite. Un abierto calibrado *sin* ningún punto por encima de 1 sería el resultado sospechoso.

Así que «$|\Gamma| > 1$» tiene tres significados posibles, y el tamaño, el patrón de signos y la repetibilidad los distinguen:

| Lo que ves | Lo que significa | Qué hacer |
|---|---|---|
| muy por encima de 1, en una banda, igual en cada repetición | error sistemático sin corregir | calibrar (o revisar la calibración) |
| ligeramente por encima y por debajo de 1, comparable a la repetibilidad | ruido en torno a un dispositivo totalmente reflectante | nada: informarlo con su incertidumbre |
| picos aislados | un problema de adquisición en esos puntos | revisar cómo se tomó el barrido |

En ninguno de los casos lo correcto es descartar la medida o recortar los valores a 1. Recortar ocultaría la evidencia y sesgaría el resultado. **Una violación de la pasividad es un hallazgo experimental, no un error de los datos.**

## 5. Poner las comprobaciones por escrito: `rfmeasurement`

Hasta ahora las comprobaciones viven en tu cabeza y en celdas dispersas. `rfmeasurement` les da una estructura. Sus ideas:

- una **`Measurement`** son los datos (un `Network` de scikit-rf) junto con su **contexto**: instrumento, calibración, dispositivo, etc.;
- una **regla de validación** examina una medida y devuelve un resultado con un **estado** (`pass`, `warning`, `fail`, `not_applicable`), la **evidencia** en la que basó la decisión y una explicación;
- las reglas se agrupan según las tres preguntas de la sección 1: `integrity.*`, `physics.*` y `quality.*`.

La biblioteca incluye una regla de pasividad, `physics.passivity`, escrita para redes de dos o más puertos; en la versión 0.3.0 no trata datos de un puerto. La dejamos fuera y escribimos nosotros la regla de un puerto, que además es la mejor manera de ver qué es una regla. Su tolerancia viene de la sección 2: 0.01 es unas tres veces la dispersión típica de un estándar totalmente reflectante.

In [ ]:
class OnePortPassivityRule(ValidationRule):
    """|Γ| no debe superar 1 más de lo que la repetibilidad puede explicar."""

    identifier = "physics.passivity_one_port"
    description = "El módulo del coeficiente de reflexión no supera 1."

    def __init__(self, tol=0.01, max_fraction=0.01):
        self.tol = tol
        self.max_fraction = max_fraction

    def is_applicable(self, measurement):
        return measurement.data.nports == 1

    def _evaluate(self, measurement):
        magnitude = np.abs(measurement.data.s[:, 0, 0])
        beyond = float(np.mean(magnitude > 1 + self.tol))
        evidence = {
            "max_magnitude": float(magnitude.max()),
            "fraction_above_1": float(np.mean(magnitude > 1)),
            "fraction_beyond_tolerance": beyond,
            "tolerance": self.tol,
        }
        if beyond > self.max_fraction:
            status = ValidationStatus.FAIL
            explanation = (
                "|Γ| supera 1 en más de la tolerancia en una parte significativa de la banda."
            )
        elif magnitude.max() > 1:
            status = ValidationStatus.WARNING
            explanation = (
                "|Γ| supera 1 solo ligeramente o en puntos aislados: "
                "compáralo con la repetibilidad."
            )
        else:
            status, explanation = ValidationStatus.PASS, None
        return ValidationResult(
            rule_id=self.identifier,
            status=status,
            description=self.description,
            evidence=evidence,
            explanation=explanation,
        )


rules = [rule for rule in DEFAULT_RULES if not isinstance(rule, PassivityRule)]
rules.append(OnePortPassivityRule())
print("Reglas:", ", ".join(rule.identifier for rule in rules))

Envuelve cada una de las tres medidas de la sección 4 y valídala. El contexto dice lo que sabemos sobre cómo se obtuvo cada una.

In [ ]:
frequency = rf.Frequency.from_f(f_hz, unit="Hz")
calibrations = {
    "abierto, en bruto": "ninguna (en bruto)",
    "abierto, calibrado": "SOL, estándares ideales, repeticiones 1-5",
    "resistencia de 75 Ω, calibrada": "SOL, estándares ideales, repeticiones 1-10",
}

measurements = {}
for name, gamma in cases.items():
    measurement = Measurement(
        data=rf.Network(frequency=frequency, s=gamma, z0=50, name=name),
        context=MeasurementContext(
            dut=name.split(",")[0],
            instrument="NanoVNA-F V2",
            calibration=calibrations[name],
            frequency_range_hz=(float(f_hz[0]), float(f_hz[-1])),
        ),
    )
    measurement.validation = validate(measurement, rules)
    measurements[name] = measurement

print(f"{'regla':<28}" + "".join(f"{name:>32}" for name in measurements))
for i, rule in enumerate(rules):
    statuses = [m.validation.results[i].status.value for m in measurements.values()]
    print(f"{rule.identifier:<28}" + "".join(f"{status:>32}" for status in statuses))

Lee la tabla por filas. Las tres reglas de **integridad** pasan en todas partes: los ficheros están bien. La reciprocidad no se aplica a un puerto. Las reglas de **calidad** pasan. Solo la fila de **física** es distinta: `fail` para el abierto en bruto, `warning` para el abierto calibrado y `pass` para la resistencia. Son las tres historias de la sección 4, ahora como registro.

Un estado por sí solo sería un semáforo. Lo que lo hace útil es la evidencia que lo acompaña:

In [ ]:
for name, measurement in measurements.items():
    result = measurement.validation.results[-1]
    print(f"{name}: {result.status.value}")
    for key, value in result.evidence.items():
        print(f"    {key}: {value:.4g}")
    if result.explanation:
        print(f"    {result.explanation}")

## 6. Trazabilidad: ¿qué necesitaría otra persona?

Supón que dentro de un año un colega lee «la resistencia es de 74.0 Ω». Para fiarse o para repetirlo, necesita saber qué instrumento, qué calibración, qué datos, qué pasos de procesado y qué software. Eso es la **trazabilidad**, y es barato registrarla en el momento e imposible reconstruirla después.

El contexto de una medida tiene un campo para cada cosa que merece la pena saber. Muéstralo para nuestra resistencia y mira cuánto falta:

In [ ]:
resistor = measurements["resistencia de 75 Ω, calibrada"]
resistor.context.averaging = len(resistor_reps)
resistor.context.confidence = {
    "instrument": MetadataConfidence.SPECIFIED,
    "calibration": MetadataConfidence.ASSUMED,
    "frequency_range_hz": MetadataConfidence.MEASURED,
}

for field in resistor.context.__dataclass_fields__:
    value = getattr(resistor.context, field)
    if field == "confidence":
        value = {key: level.value for key, level in value.items()}
    known = value is not None and value != {}
    print(f"{field:<20} {value if known else '— desconocido'}")

Operador, cables, utillaje, temperatura, potencia, ancho de banda, fecha: todo desconocido. No es una crítica a este conjunto de datos en particular; es el aspecto que tienen casi todos los ficheros de medida unos meses después. Fíjate también en la entrada `confidence`: el instrumento está *especificado* (nos lo dijeron), el intervalo de frecuencias está *medido* (está en los datos) y la calibración está *supuesta* (estándares ideales). Un registro honesto separa lo que se midió de lo que se tomó por bueno.

La otra mitad de la trazabilidad es el historial de procesado: cada paso que convirtió los barridos en bruto en el número que se informa. Registra los pasos y pide después a `rfmeasurement` un informe del resultado de la sección 3 (el informe lo genera la biblioteca, en inglés).

In [ ]:
software = f"vnacourse {vnacourse.__version__}"
averaged = ProvenanceRecord(
    operation="promediar repeticiones",
    software_version=software,
    parameters={"repetitions": len(resistor_reps)},
    inputs=("data/dut/load_75ohm",),
)
calibrated = ProvenanceRecord(
    operation="calibración SOL",
    software_version=software,
    parameters={"standards": "supuestos ideales", "repetitions_averaged": len(open_reps)},
    inputs=(averaged.record_id, "data/standards"),
)
resistor.provenance = [averaged, calibrated]

report = resistor.validation
if report.has_failures:
    overall = ValidationStatus.FAIL
elif report.has_warnings:
    overall = ValidationStatus.WARNING
else:
    overall = ValidationStatus.PASS

result = AnalysisResult(
    measurand=Measurand(
        name="Resistencia por debajo de 10 MHz",
        definition="media de Re Z en los puntos de frecuencia por debajo de 10 MHz",
        unit="ohm",
    ),
    value=round(float(resistance), 2),
    unit="ohm",
    standard_uncertainty=round(float(u_combined), 2),
    provenance=(averaged, calibrated),
    validation_status=overall,
)

Markdown(generate_report(resistor, result))

Una página reúne el valor, su incertidumbre, qué comprobaciones se hicieron y cómo salieron, las versiones del software y un nivel de reproducibilidad. El nivel que se muestra es el más alto porque hemos aportado el instrumento, la calibración y un historial de procesado. No juzga si son *buenos*: una calibración supuesta ideal cuenta como registrada. El informe completa el registro; el juicio de las secciones 3 y 4 sigue siendo tuyo.

## Comprueba lo que has entendido

1. Diez repeticiones dan una desviación típica de 0.003. ¿Cuál es la incertidumbre típica de su media? ¿Cuántas repeticiones necesitarías para reducirla a la mitad?

<details><summary>Respuesta</summary>

$u = 0.003/\sqrt{10} \approx 0.001$. La incertidumbre cae como $1/\sqrt{n}$, así que reducirla a la mitad exige cuatro veces más: 40 repeticiones.
</details>

2. Promediar más repeticiones reduce la incertidumbre de 0.2 Ω de la sección 3. ¿Acerca eso el resultado a la resistencia real?

<details><summary>Respuesta</summary>

Solo en lo que respecta al error aleatorio. Si el estándar de carga no es exactamente de 50 Ω, todas las calibraciones se equivocan de la misma manera y el promedio converge, cada vez con más precisión, al valor equivocado. La repetición reduce el error aleatorio; no hace nada con el sistemático.
</details>

3. Un colega propone limpiar los datos sustituyendo todo $|\Gamma| > 1$ por exactamente 1. ¿Qué tiene de malo?

<details><summary>Respuesta</summary>

En el abierto calibrado el ruido se dispersa a ambos lados de 1. Recortar solo el lado superior lleva el promedio por debajo de 1: introduce un sesgo. Además destruye la evidencia: después nadie puede saber si las excursiones eran ruido, un pico de adquisición o un problema de calibración.
</details>

4. ¿Qué estado de validación asignarías a un fichero en el que dos puntos de frecuencia están desordenados: integridad, física o calidad? ¿Lo tratarías como el fallo de pasividad del abierto en bruto?

<details><summary>Respuesta</summary>

Integridad. Es un fallo de otra naturaleza: la propia estructura de los datos es errónea, así que nada de lo que se calcule con ellos es fiable hasta corregirla. Un fallo de física en un fichero íntegro es un hallazgo sobre la medida y debe conservarse y explicarse.
</details>

## Ejercicios

Las soluciones resueltas están en [`exercises/solutions/05_solutions.ipynb`](../exercises/solutions/05_solutions.ipynb).

1. **¿Media o mediana?** Vuelve a calcular la calibración de la sección 4 (repeticiones 1–5) usando la mediana entre repeticiones en lugar de la media (toma la mediana de la parte real y de la imaginaria por separado). Corrige la media y la mediana de las repeticiones 6–10 del abierto y compara la distancia típica y la del peor caso a $\Gamma = 1$. ¿Qué estadístico trata mejor los picos?
2. **Incertidumbre en toda la banda.** La sección 3 solo miró por debajo de 10 MHz. Calcula a cada frecuencia la dispersión del $\Gamma$ calibrado de la resistencia de 75 Ω entre las diez calibraciones de una sola repetición, y represéntala. ¿Es la calibración igual de repetible en toda la banda?
3. **Escribe una regla.** Escribe una `ValidationRule` llamada `quality.spikes` que avise cuando un único barrido tenga puntos que difieran de la mediana de sus vecinos en más de un umbral. Aplícala a una repetición del abierto en bruto e indica dónde están los picos.
4. **Tu propio contexto.** Rellena un `MeasurementContext` para una medida que pudieras tomar en tu laboratorio. Para cada campo, decide si su valor sería *medido*, *especificado*, *estimado* o *supuesto*.

In [ ]:
# Tu trabajo aquí

## Resumen

- Los barridos repetidos dan la repetibilidad de una medida. Aquí es de unos 0.003 para dispositivos totalmente reflectantes y de 0.0005 para uno adaptado, con picos ligados a la adquisición.
- La incertidumbre típica de la media de $n$ lecturas es $s/\sqrt{n}$. Solo cubre el error aleatorio; di qué deja fuera.
- La pasividad, $|\Gamma| \le 1$, es una prueba física que no necesita referencia. Una violación es evidencia: de un error sistemático si es grande y repetible, de ruido si es tan pequeña como la repetibilidad.
- Integridad, coherencia física y calidad son preguntas distintas con consecuencias distintas.
- Un resultado es trazable cuando su contexto y su historial de procesado se registran con él.

**Este es el último cuaderno del módulo básico.** El cuaderno 04, sobre la adquisición de tus propias medidas con un NanoVNA, es opcional, necesita el instrumento y todavía no está disponible.